In [ ]:
pip install pdfplumber

In [8]:
import os
import json
import pandas as pd
import pdfplumber
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [9]:
# ---------- Step 1: Setup LLM Agent Pipeline ----------

# Setup LLM agent pipeline
llm = ChatOpenAI(model="gpt-4o")

# Define the prompt template
prompt_template = ChatPromptTemplate.from_template("""
Extract the following vehicle specifications from the text and return as valid JSON:
- Vehicle Name
- Torque
- Engine Capacity
- CO2 Emissions
- Price
- Fuel Type
- Fuel Consumption (Combined)
- Dimensions (L x W x H)
- Suspension
- Ambient Lighting
- Key Features

Text: {brochure_text}

Respond ONLY with valid JSON output.
""")

pipeline = prompt_template | llm

In [6]:
# ---------- Step 2: Function to extract text from PDF ----------
def extract_text_from_pdf(pdf_path):
    text = ""
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            text += page.extract_text() + "\n"
    return text

In [11]:
# ---------- Step 3: Process Multiple PDFs ----------

# Function to extract relevant text from selected pages (e.g., last 5 pages for specs)
def extract_relevant_text_from_pdf(pdf_path, max_pages=5):
    text = ""
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages[-max_pages:]:  # Use only last 'max_pages' pages
            text += page.extract_text() + "\n"
    return text

# Function to chunk text to reduce token size
def chunk_text(text, chunk_size=1500):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=200,
    )
    return splitter.split_text(text)

In [ ]:
# Process multiple PDFs
brochure_folder = "/Users/surendra/ai_agents/JLR_PDF"

extracted_data_list = []

for filename in os.listdir(brochure_folder):
    if filename.endswith(".pdf"):
        filepath = os.path.join(brochure_folder, filename)
        print(f"Processing: {filename}")

        brochure_text = extract_relevant_text_from_pdf(filepath, max_pages=5)
        chunks = chunk_text(brochure_text, chunk_size=1500)

        combined_output = {}

        for chunk in chunks:
            result = pipeline.invoke({"brochure_text": chunk})
            try:
                extracted_data = json.loads(result.content)

                # Merge chunks - if key exists, prefer first non-empty value
                for key, value in extracted_data.items():
                    if key not in combined_output or not combined_output[key]:
                        combined_output[key] = value

            except json.JSONDecodeError:
                print(f"Warning: JSON parse error in {filename} chunk")
                continue

        combined_output["Source File"] = filename
        extracted_data_list.append(combined_output)

In [16]:
df = pd.DataFrame(extracted_data_list)

df.to_csv("vehicle_specs_output.csv", index=False)

df.to_json("vehicle_specs_output.json", orient="records", indent=4)

print("✅ Extraction Completed. Files saved as vehicle_specs_output.csv and vehicle_specs_output.json")

✅ Extraction Completed. Files saved as vehicle_specs_output.csv and vehicle_specs_output.json


In [13]:
# Save results
df = pd.DataFrame(extracted_data_list)

df.to_csv("vehicle_specs_output_reduced.csv", index=False)

df.to_json("vehicle_specs_output_reduced.json", orient="records", indent=4)

print("✅ Extraction Completed with Reduced Tokens.")


✅ Extraction Completed with Reduced Tokens.


In [ ]:
# ---------- Step 5: Provide Recommendations ----------

print("\n Recommendations:")

for vehicle in extracted_data_list:
    price = vehicle.get("Price", "").replace("£", "").replace(",", "").strip()
    try:
        price_value = float(price)
    except:
        price_value = None

    print(f"\n🔹 {vehicle.get('Vehicle Name', 'Unknown Model')}:")

    # Upselling suggestion
    if price_value and price_value > 150000:
        print("   - Premium vehicle: Recommend luxury packages & concierge services.")
    elif price_value and price_value < 80000:
        print("   - Entry segment: Offer extended warranty & maintenance plans.")
    else:
        print("   - Mid-range: Suggest feature upgrades like ambient lighting or air suspension.")

    # Environmental suggestion
    if "313 g/km" in vehicle.get("CO2 Emissions", ""):
        print("   - High CO2: Recommend hybrid or electric alternatives for eco-conscious customers.")


In [18]:
%%writefile vehicle_agentic_pipeline.py

import os
import json
import pandas as pd
import pdfplumber
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Setup LLM
llm = ChatOpenAI(model="gpt-4o")

# Prompt for extracting vehicle specs
spec_prompt = ChatPromptTemplate.from_template("""
Vehicle Name: {vehicle_name}

Extract these specifications from the text and respond as valid JSON:
- Vehicle Name (Use the given vehicle name if not explicitly mentioned)
- Torque
- Engine Capacity
- CO2 Emissions
- Price
- Fuel Type
- Fuel Consumption (Combined)
- Dimensions (L x W x H)
- Suspension
- Ambient Lighting
- Key Features

Text: {brochure_text}

Respond ONLY with valid JSON output.
""")

spec_pipeline = spec_prompt | llm

# Prompt for extracting vehicle name
def extract_vehicle_name(text):
    name_prompt = ChatPromptTemplate.from_template("""
Identify the vehicle name or model from the text. Respond ONLY with the model name as plain text.

Text: {brochure_text}
""")
    name_pipeline = name_prompt | llm
    result = name_pipeline.invoke({"brochure_text": text})
    return result.content.strip()

# Function to extract text from PDF
def extract_relevant_text_from_pdf(pdf_path, max_pages=5):
    text = ""
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages[-max_pages:]:
            text += page.extract_text() + "\n"
    return text

# Function to chunk text
def chunk_text(text, chunk_size=1500):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=200,
    )
    return splitter.split_text(text)

# Process all PDFs
brochure_folder = "/Users/surendra/ai_agents/JLR_PDF"
extracted_data_list = []

for filename in os.listdir(brochure_folder):
    if filename.endswith(".pdf"):
        filepath = os.path.join(brochure_folder, filename)
        print(f"Processing: {filename}")

        full_text = extract_relevant_text_from_pdf(filepath, max_pages=5)
        vehicle_name = extract_vehicle_name(full_text)

        chunks = chunk_text(full_text, chunk_size=1500)
        combined_output = {}

        for chunk in chunks:
            result = spec_pipeline.invoke({"brochure_text": chunk, "vehicle_name": vehicle_name})
            try:
                extracted_data = json.loads(result.content)
                for key, value in extracted_data.items():
                    if key not in combined_output or not combined_output[key]:
                        combined_output[key] = value
            except json.JSONDecodeError:
                print(f"Warning: JSON parse error in {filename} chunk")
                continue

        combined_output["Source File"] = filename
        extracted_data_list.append(combined_output)

# Save output
output_df = pd.DataFrame(extracted_data_list)
output_df.to_csv("vehicle_specs_final.csv", index=False)
output_df.to_json("vehicle_specs_final.json", orient="records", indent=4)

print("✅ Extraction Completed with Proper Vehicle Names.")

Writing vehicle_agentic_pipeline.py


In [ ]:
!python3 vehicle_agentic_pipeline.py

In [24]:
%%writefile vehicle_agentic_pipeline1.py

import os
import json
import pandas as pd
import pdfplumber
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Setup LLM
llm = ChatOpenAI(model="gpt-4o")

# Define unified prompt for vehicle specs extraction
prompt_template = ChatPromptTemplate.from_template("""
Extract the following vehicle specifications and output valid JSON.

If a field is missing, set it to "N/A".

ALWAYS respond with valid JSON.

Fields:
- Vehicle Name
- Torque
- Engine Capacity
- CO2 Emissions
- Price
- Fuel Type
- Fuel Consumption (Combined)
- Dimensions (L x W x H)
- Suspension
- Ambient Lighting
- Key Features

Text: {brochure_text}
""")

# Create pipeline
pipeline = prompt_template | llm

# Function to extract text from PDF
def extract_relevant_text_from_pdf(pdf_path, max_pages=5):
    text = ""
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages[-max_pages:]:
            text += page.extract_text() + "\n"
    return text

# Function to chunk text
def chunk_text(text, chunk_size=1500):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=200,
    )
    return splitter.split_text(text)

# Process all PDFs
brochure_folder = "/Users/surendra/ai_agents/JLR_PDF"

extracted_data_list = []

for filename in os.listdir(brochure_folder):
    if filename.endswith(".pdf"):
        filepath = os.path.join(brochure_folder, filename)
        print(f"Processing: {filename}")

        full_text = extract_relevant_text_from_pdf(filepath, max_pages=5)
        chunks = chunk_text(full_text, chunk_size=1500)
        combined_output = {}

        for chunk in chunks:
            result = pipeline.invoke({"brochure_text": chunk})
            try:
                extracted_data = json.loads(result.content)
                for key, value in extracted_data.items():
                    if key not in combined_output or not combined_output[key]:
                        combined_output[key] = value
            except json.JSONDecodeError:
                print(f"Warning: JSON parse error in {filename} chunk")
                continue

        combined_output["Source File"] = filename
        extracted_data_list.append(combined_output)

# Save output
output_df = pd.DataFrame(extracted_data_list)
print(output_df.head(10))

#output_df.to_csv("vehicle_specs_final.csv", index=False)
#output_df.to_json("vehicle_specs_final.json", orient="records", indent=4)

print("✅ Extraction Completed with Unified Prompt.")

Overwriting vehicle_agentic_pipeline1.py


In [ ]:
file_path='/Users/surendra/ai_agents/walmart_sales.xlsx'

In [ ]:
!python3 vehicle_agentic_pipeline1.py

In [26]:
#Example: Relevant Text Extraction Code

#Prompt for Relevant Text Extraction

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

llm = ChatOpenAI(model="gpt-4o")

# Plain text extraction prompt
text_prompt = ChatPromptTemplate.from_template("""
Extract ONLY the relevant sections from the brochure text related to:

- Technical Specifications
- Dimensions
- Engine Capacity
- Torque
- Suspension
- Features like Ambient Lighting, Comfort Seats

Do not include price, warranty, or legal information.

Return the extracted text as plain text.

Text: {brochure_text}
""")

pipeline = text_prompt | llm

In [ ]:
#Processing the PDFs
import os
import pdfplumber

def extract_pdf_text(pdf_path, max_pages=5):
    text = ""
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages[-max_pages:]:
            page_text = page.extract_text()
            if page_text:
                text += page_text + "\n"
    return text

brochure_folder = "/Users/surendra/ai_agents/JLR_PDF"
output_texts = {}

for filename in os.listdir(brochure_folder):
    if filename.endswith(".pdf"):
        filepath = os.path.join(brochure_folder, filename)
        print(f"Processing {filename}")

        brochure_text = extract_pdf_text(filepath, max_pages=5)

        result = pipeline.invoke({"brochure_text": brochure_text})

        # Save the relevant text to file
        output_texts[filename] = result.content
        with open(f"extracted_specs_{filename.replace('.pdf', '.txt')}", "w") as f:
            f.write(result.content)

print("✅ Relevant text extracted successfully.")

In [28]:
%%writefile vehicle_agentic_pipeline2.py

import os
import json
import pandas as pd
import pdfplumber
import streamlit as st
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Setup LLM
llm = ChatOpenAI(model="gpt-4o")

# Define unified prompt for vehicle specs extraction
prompt_template = ChatPromptTemplate.from_template("""
Extract the following vehicle specifications from the text and return as valid JSON:
- Vehicle Name
- Torque
- Engine Capacity
- CO2 Emissions
- Price
- Fuel Type
- Fuel Consumption (Combined)
- Dimensions (L x W x H)
- Suspension
- Ambient Lighting
- Key Features

Text: {brochure_text}

Respond ONLY with valid JSON output.
""")

# Create pipeline
pipeline = prompt_template | llm

# Function to extract text from PDF
def extract_relevant_text_from_pdf(pdf_path, max_pages=5):
    text = ""
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages[-max_pages:]:
            text += page.extract_text() + "\n"
    return text

# Function to chunk text
def chunk_text(text, chunk_size=1500):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=200,
    )
    return splitter.split_text(text)

# Process all PDFs and generate relevant text .txt files
brochure_folder = "/Users/surendra/ai_agents/JLR_PDF"
extracted_data_list = []

for filename in os.listdir(brochure_folder):
    if filename.endswith(".pdf"):
        filepath = os.path.join(brochure_folder, filename)
        print(f"Processing: {filename}")

        full_text = extract_relevant_text_from_pdf(filepath, max_pages=5)
        chunks = chunk_text(full_text, chunk_size=1500)
        combined_output = {}
        extracted_text = ""

        for chunk in chunks:
            result = pipeline.invoke({"brochure_text": chunk})
            try:
                extracted_data = json.loads(result.content)
                for key, value in extracted_data.items():
                    if key not in combined_output or not combined_output[key]:
                        combined_output[key] = value
            except json.JSONDecodeError:
                # Save plain text from chunk to extracted text
                extracted_text += chunk + "\n"
                continue

        combined_output["Source File"] = filename
        extracted_data_list.append(combined_output)

        # Save extracted text to .txt file
        txt_filename = f"extracted_specs_{filename.replace('.pdf', '.txt')}"
        with open(txt_filename, "w") as f:
            f.write(extracted_text)

# Save structured output
output_df = pd.DataFrame(extracted_data_list)
output_df.to_csv("vehicle_specs_final.csv", index=False)
output_df.to_json("vehicle_specs_final.json", orient="records", indent=4)

# Generate report from .txt files
report_data = []

for file in os.listdir("./"):
    if file.startswith("extracted_specs_") and file.endswith(".txt"):
        with open(file, "r") as f:
            content = f.read()
        report_entry = {
            "Source File": file.replace("extracted_specs_", "").replace(".txt", ".pdf"),
            "Extracted Specs (Preview)": content[:500] + "..." if len(content) > 500 else content
        }
        report_data.append(report_entry)

report_df = pd.DataFrame(report_data)
report_df.to_csv("vehicle_specs_text_extraction_report.csv", index=False)

print("✅ Extraction and report generation completed successfully.")

# Streamlit Viewer for the Report
st.set_page_config(page_title="Vehicle Specs Report Viewer", layout="wide")
st.title("🚗 Vehicle Specs Text Extraction Report")

st.header("Summary Report")

st.dataframe(report_df, use_container_width=True)

st.download_button(
    label="Download Report as CSV",
    data=report_df.to_csv(index=False),
    file_name="vehicle_specs_text_extraction_report.csv",
    mime="text/csv"
)

Writing vehicle_agentic_pipeline2.py


In [ ]:
pip install streamlit

In [ ]:
!streamlit run vehicle_agentic_pipeline2.py

In [3]:
%%writefile vehicle_data_extraction.py

import os
import json
import pandas as pd
import pdfplumber
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Setup LLM
llm = ChatOpenAI(model="gpt-4o")

# Define unified prompt for vehicle specs extraction
prompt_template = ChatPromptTemplate.from_template("""
Extract the following vehicle specifications from the text and return as valid JSON:
- Vehicle Name
- Torque
- Engine Capacity
- CO2 Emissions
- Price
- Fuel Type
- Fuel Consumption (Combined)
- Dimensions (L x W x H)
- Suspension
- Ambient Lighting
- Key Features

Text: {brochure_text}

Respond ONLY with valid JSON output.
""")

# Create pipeline
pipeline = prompt_template | llm

# Function to extract text from PDF
def extract_relevant_text_from_pdf(pdf_path, max_pages=5):
    text = ""
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages[-max_pages:]:
            text += page.extract_text() + "\n"
    return text

# Function to chunk text
def chunk_text(text, chunk_size=1500):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=200,
    )
    return splitter.split_text(text)

# Process all PDFs and generate relevant text .txt files
brochure_folder = "/Users/surendra/ai_agents/JLR_PDF"
extracted_data_list = []

for filename in os.listdir(brochure_folder):
    if filename.endswith(".pdf"):
        filepath = os.path.join(brochure_folder, filename)
        print(f"Processing: {filename}")

        full_text = extract_relevant_text_from_pdf(filepath, max_pages=5)
        chunks = chunk_text(full_text, chunk_size=1500)
        combined_output = {}
        extracted_text = ""

        for chunk in chunks:
            result = pipeline.invoke({"brochure_text": chunk})
            try:
                extracted_data = json.loads(result.content)
                for key, value in extracted_data.items():
                    if key not in combined_output or not combined_output[key]:
                        combined_output[key] = value
            except json.JSONDecodeError:
                extracted_text += chunk + "\n"
                continue

        combined_output["Source File"] = filename
        extracted_data_list.append(combined_output)

        # Save extracted text to .txt file
        txt_filename = f"extracted_specs_{filename.replace('.pdf', '.txt')}"
        with open(txt_filename, "w") as f:
            f.write(extracted_text)

# Save structured output
output_df = pd.DataFrame(extracted_data_list)
output_df.to_csv("vehicle_specs_final.csv", index=False)
output_df.to_json("vehicle_specs_final.json", orient="records", indent=4)

# Generate report from .txt files
report_data = []
summary_text = ""

for file in os.listdir("./"):
    if file.startswith("extracted_specs_") and file.endswith(".txt"):
        with open(file, "r") as f:
            content = f.read()
        report_entry = {
            "Source File": file.replace("extracted_specs_", "").replace(".txt", ".pdf"),
            "Extracted Specs (Preview)": content[:500] + "..." if len(content) > 500 else content
        }
        report_data.append(report_entry)
        summary_text += content + "\n"

report_df = pd.DataFrame(report_data)
report_df.to_csv("vehicle_specs_text_extraction_report.csv", index=False)

# Generate Summary Report using LLM
summary_prompt = f"""
You are a Vehicle Reporting Assistant. Create a concise summary report based on the following extracted specifications:
{summary_text}
"""

summary_result = llm.invoke(summary_prompt)

with open("vehicle_summary_report.txt", "w") as f:
    f.write(summary_result.content)

print("✅ Data extraction, summary generation, and report creation completed.")

Writing vehicle_data_extraction.py


In [ ]:
!python3 vehicle_data_extraction.py

In [44]:
#multi_agent_pdf_pipeline.py

import os
import pdfplumber
import json
import pandas as pd
from autogen import AssistantAgent, UserProxyAgent, GroupChat, GroupChatManager
from langchain.text_splitter import RecursiveCharacterTextSplitter

# LLM config
config = {
    "model": "gpt-4o",
    "api_key": os.getenv("OPENAI_API_KEY")
}

In [45]:
# Define Agents
extractor = AssistantAgent(
    name="Extractor",
    system_message="You are a vehicle data extractor. Extract Torque, Engine, CO2, Price, Fuel, Dimensions, Suspension, Ambient Lighting from input text. Output valid JSON.",
    llm_config=config
)

In [46]:
structurer = AssistantAgent(
    name="Structurer",
    system_message="You are a data structurer. Merge multiple JSONs from extracted data into a single JSON with all fields filled. Use 'N/A' if data is missing.",
    llm_config=config
)

In [47]:
decision_maker = AssistantAgent(
    name="DecisionAssistant",
    system_message="You are a decision assistant. Given vehicle specs, write a human-friendly report with comparison, upselling, and eco-friendly suggestions.",
    llm_config=config
)

In [48]:
user = UserProxyAgent(
    name="User",
    code_execution_config={
        "use_docker": False },
    human_input_mode="NEVER",  # Simulates user
    max_consecutive_auto_reply=2
)

In [49]:
# GroupChat setup
group_chat = GroupChat(
    agents=[user, extractor, structurer, decision_maker],
    messages=[],
)

manager = GroupChatManager(
    groupchat=group_chat,
    llm_config=config
)

In [50]:
# Function to extract relevant text from last N pages
def extract_relevant_text_from_pdf(pdf_path, max_pages=5):
    text = ""
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages[-max_pages:]:
            if page.extract_text():
                text += page.extract_text() + "\n"
    return text

# Function to chunk text into manageable pieces
def chunk_text(text, chunk_size=1500, overlap=200):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=overlap
    )
    return splitter.split_text(text)

In [51]:
# Input folder
brochure_folder = "/Users/surendra/ai_agents/JLR_PDF"
extracted_data_list = []
recommendations_list = []

In [ ]:
for filename in os.listdir(brochure_folder):
    if filename.endswith(".pdf"):
        filepath = os.path.join(brochure_folder, filename)
        print(f"Processing: {filename}")

        # Extract and chunk relevant text
        full_text = extract_relevant_text_from_pdf(filepath, max_pages=5)
        chunks = chunk_text(full_text, chunk_size=1500)

        combined_output = {}
        extracted_text = ""
        rec_text = ""

        for chunk in chunks:
            # Initiate multi-agent conversation per chunk
            user.initiate_chat(
                manager,
                message=f"Process this vehicle brochure chunk:\n{chunk}"
            )

            # Collect multi-agent outputs
            all_msgs = group_chat.messages

            for msg in all_msgs:
                if msg["role"] == "assistant":
                    content = msg["content"]
                    try:
                        # Attempt JSON extraction
                        data = json.loads(content)
                        for key, value in data.items():
                            if key not in combined_output or not combined_output[key]:
                                combined_output[key] = value
                    except json.JSONDecodeError:
                        # Capture fallback recommendations/text output
                        rec_text += content + "\n"

            # Reset group chat for next chunk
            group_chat.messages = []

        combined_output["Source File"] = filename
        extracted_data_list.append(combined_output)

        # Save extracted recommendations or fallback text
        txt_filename = f"extracted_specs_{filename.replace('.pdf', '.txt')}"
        with open(txt_filename, "w") as f:
            f.write(rec_text)

# Save structured specs to CSV
df_specs = pd.DataFrame(extracted_data_list)
df_specs.to_csv("autogen_vehicle_specs_chunkwise.csv", index=False)

print("AutoGen Chunk-wise Multi-Agent PDF Processing Completed.")